# Mini Project Phase 1 — Group 8: Retail Grocery Inventory & POS Warehouse

**Dataset:** `Grocery_Inventory_and_Sales_Dataset.csv` (990 rows × 16 columns)

**Business scenario:** A supermarket chain needs to monitor daily store sales, stock turnover rates, and inventory reorder thresholds across multiple branches.

**Pipeline:** process the messy POS extract → clean currency symbols / dates / headers → load a star-schema SQLite warehouse → store derived supplier audit logs in MongoDB → answer monthly sales variance questions with SQL CTEs.

All heavy lifting lives in `grocery_etl.py`; this notebook narrates and verifies each stage.

In [1]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

import grocery_etl as etl

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 25)

RAW_CSV = Path('Grocery_Inventory_and_Sales_Dataset.csv')
DB_PATH = Path('grocery_warehouse.db')
raw = pd.read_csv(RAW_CSV, dtype=str)
print(raw.shape)
raw.head(3)

(990, 16)


,Product_ID,Product_Name,Catagory,Supplier_ID,Supplier_Name,Stock_Quantity,Reorder_Level,Reorder_Quantity,Unit_Price,Date_Received,Last_Order_Date,Expiration_Date,Warehouse_Location,Sales_Volume,Inventory_Turnover_Rate,Status
0,29-205-1132,Sushi Rice,Grains & Pulses,38-037-1699,Jaxnation,22,72,70,$4.50,8/16/2024,6/29/2024,9/19/2024,48 Del Sol Trail,32,19,Discontinued
1,40-681-9981,Arabica Coffee,Beverages,54-470-2479,Feedmix,45,77,2,$20.00,11/1/2024,5/29/2024,5/8/2024,36 3rd Place,85,1,Discontinued
2,06-955-3428,Black Rice,Grains & Pulses,54-031-2945,Vinder,30,38,83,$6.00,8/3/2024,6/10/2024,9/22/2024,3296 Walton Court,31,34,Backordered


## 1. Data Profiling — what's messy?

Profiling the raw extract surfaces four data-quality issues the pipeline must fix:

| Issue | Evidence | Fix |
|---|---|---|
| Currency symbols in price | `Unit_Price` = `"$4.50 "` on all 990 rows | strip `$`, commas, whitespace → float |
| Misspelled header | `Catagory` | rename → `category` |
| Missing category | 1 null | impute `'Uncategorized'` + audit event |
| Mixed-width dates | `M/D/YYYY`, `MM/DD/YYYY`, `M/DD/YYYY` | parse → ISO `YYYY-MM-DD` |
| Unvalidated IDs | `XX-XXX-XXXX` pattern assumed | regex-validate, quarantine failures |

In [2]:
print('Unit_Price samples :', raw['Unit_Price'].unique()[:5])
print('Missing per column :')
print(raw.isna().sum()[lambda s: s > 0])
print()
print('Date format widths (Date_Received):')
print(raw['Date_Received'].str.replace(r'\d', '#', regex=True).value_counts())

Unit_Price samples : <StringArray>
['$4.50 ', '$20.00 ', '$6.00 ', '$1.50 ', '$4.00 ']
Length: 5, dtype: str
Missing per column :
Catagory    1
dtype: int64

Date format widths (Date_Received):
Date_Received
#/##/####     511
#/#/####      225
##/##/####    184
##/#/####      70
Name: count, dtype: int64


## 2. Cleansing

`etl.transform()` applies the rules above, validates ID patterns, drops exact duplicates, quarantines unrecoverable rows to `quarantined_rows.csv`, and stamps each row with a SHA-256 `row_hash` for idempotent loads.

In [3]:
clean, quarantine, metrics = etl.transform(raw)
print(json.dumps(metrics, indent=2))
clean[['product_name','category','unit_price','last_order_date','stock_quantity','sales_volume','status']].head(5)

[TRANSFORM] 990/990 rows clean; 0 quarantined; 0 dupes; 990 currency symbols stripped; 1 missing categories filled.


{
  "raw_rows": 990,
  "price_symbols_cleaned": 990,
  "missing_category_filled": 1,
  "duplicates_dropped": 0,
  "quarantined": 0,
  "cleaned_rows": 990
}


,product_name,category,unit_price,last_order_date,stock_quantity,sales_volume,status
0,Sushi Rice,Grains & Pulses,4.5,2024-06-29,22,32,Discontinued
1,Arabica Coffee,Beverages,20.0,2024-05-29,45,85,Discontinued
2,Black Rice,Grains & Pulses,6.0,2024-06-10,30,31,Backordered
3,Long Grain Rice,Grains & Pulses,1.5,2025-02-19,12,95,Active
4,Plum,Fruits & Vegetables,4.0,2024-10-11,37,62,Backordered


## 3. Load — SQLite star schema

The raw `product_id` / `supplier_id` are record-level (unique per row), so the **business entities** become dimension keys: `dim_product` keyed on `(product_name, category)`, `dim_supplier` on `supplier_name`, `dim_store` on `warehouse_location`. `fact_inventory` keeps the raw IDs as degenerate references plus all measures.

```
              dim_product          dim_supplier          dim_store
             (124 rows)            (350 rows)           (990 rows)
                  \                    |                    /
                   \                   |                   /
                        fact_inventory (990 rows)
```

Two views are built on top for the expected outcome — automated replenishment and stockout prevention:

- `v_reorder_recommendations` — items at/below reorder level with suggested order qty + priority
- `v_stockout_risk` — days-of-cover (on-hand ÷ daily sales velocity) tiered CRITICAL/HIGH/WATCH/LOW

In [4]:
dim_product, dim_supplier, dim_store, fact = etl.build_star_schema(clean)
counts = etl.load_sqlite(DB_PATH, dim_product, dim_supplier, dim_store, fact, full_refresh=True)
print(json.dumps(counts, indent=2))

[LOAD - SQLITE] fact_inventory=990, dim_product=124, dim_supplier=350, dim_store=990


{
  "fact_inventory": 990,
  "dim_product": 124,
  "dim_supplier": 350,
  "dim_store": 990,
  "reorder_flags": 298,
  "stockout_critical": 27
}


In [5]:
conn = sqlite3.connect(DB_PATH)
pd.read_sql('''
    SELECT f.record_key, p.product_name, p.category, s.supplier_name,
           st.warehouse_location, f.stock_quantity, f.unit_price,
           f.sales_volume, f.last_order_date, f.status
    FROM fact_inventory f
    JOIN dim_product  p  ON p.product_key  = f.product_key
    JOIN dim_supplier s  ON s.supplier_key = f.supplier_key
    JOIN dim_store    st ON st.store_key   = f.store_key
    LIMIT 5
''', conn)

,record_key,product_name,category,supplier_name,warehouse_location,stock_quantity,unit_price,sales_volume,last_order_date,status
0,1,Sushi Rice,Grains & Pulses,Jaxnation,48 Del Sol Trail,22,4.5,32,2024-06-29,Discontinued
1,2,Arabica Coffee,Beverages,Feedmix,36 3rd Place,45,20.0,85,2024-05-29,Discontinued
2,3,Black Rice,Grains & Pulses,Vinder,3296 Walton Court,30,6.0,31,2024-06-10,Backordered
3,4,Long Grain Rice,Grains & Pulses,Brightbean,3 Westerfield Crossing,12,1.5,95,2025-02-19,Active
4,5,Plum,Fruits & Vegetables,Topicstorm,15068 Scoville Court,37,4.0,62,2024-10-11,Backordered


## 4. Monthly Sales Variance — SQL CTEs

Each row is an inventory snapshot, so monthly sales are attributed to the `last_order_date` month with `revenue = sales_volume × unit_price`. `monthly_variance.sql` uses a `monthly_sales` CTE + `LAG()` window for month-over-month variance — the same pattern as the Lesson 5 `MonthlyStoreRevenue` exercise.

In [6]:
monthly = pd.read_sql('''
    WITH monthly_sales AS (
        SELECT strftime('%Y-%m', last_order_date)        AS sales_month,
               COUNT(*)                                  AS products_sold,
               SUM(sales_volume)                         AS units_sold,
               ROUND(SUM(sales_volume * unit_price), 2)  AS revenue
        FROM fact_inventory
        GROUP BY sales_month
    )
    SELECT sales_month, products_sold, units_sold, revenue,
           ROUND(revenue - LAG(revenue) OVER (ORDER BY sales_month), 2) AS mom_variance,
           ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY sales_month))
                 / LAG(revenue) OVER (ORDER BY sales_month), 1)        AS mom_variance_pct
    FROM monthly_sales ORDER BY sales_month
''', conn)
monthly

,sales_month,products_sold,units_sold,revenue,mom_variance,mom_variance_pct
0,2024-02,22,1347,12016.20,NaN,NaN
1,2024-03,77,4603,19695.08,7678.88,63.9
2,2024-04,70,4051,30830.68,11135.60,56.5
3,2024-05,84,4961,26052.07,-4778.61,-15.5
4,2024-06,98,5754,35664.85,9612.78,36.9
5,2024-07,96,5619,32055.02,-3609.83,-10.1
6,2024-08,78,4465,30580.00,-1475.02,-4.6
7,2024-09,60,3720,17941.30,-12638.70,-41.3
8,2024-10,93,5608,38442.25,20500.95,114.3
9,2024-11,71,4345,26322.90,-12119.35,-31.5


In [7]:
# MoM variance per category — worst and best months
cat = pd.read_sql('''
    WITH category_monthly AS (
        SELECT p.category,
               strftime('%Y-%m', f.last_order_date)          AS sales_month,
               ROUND(SUM(f.sales_volume * f.unit_price), 2) AS revenue
        FROM fact_inventory f
        JOIN dim_product p ON p.product_key = f.product_key
        GROUP BY p.category, sales_month
    )
    SELECT category, sales_month, revenue,
           ROUND(100.0 * (revenue - LAG(revenue) OVER (
                    PARTITION BY category ORDER BY sales_month))
                 / NULLIF(LAG(revenue) OVER (
                    PARTITION BY category ORDER BY sales_month), 0), 1) AS mom_pct
    FROM category_monthly
''', conn)
ranked = cat.sort_values('mom_pct').dropna()
pd.concat([ranked.head(3), ranked.tail(3)])

,category,sales_month,revenue,mom_pct
10,Bakery,2024-12,300.00,-79.6
38,Dairy,2025-02,938.10,-75.7
8,Bakery,2024-10,507.50,-74.1
21,Beverages,2024-10,11463.30,299.6
11,Bakery,2025-01,1667.30,455.8
53,Grains & Pulses,2024-03,1909.75,943.6


## 5. Supplier Audit Logs → MongoDB

One audit document per supplier entity (350 total) is derived deterministically from the cleaned data. Each document embeds an `audit_events` array — this is why MongoDB is the right target: the event list is variable-length and semi-structured.

| Event type | Severity | Trigger |
|---|---|---|
| `REORDER_TRIGGERED` | HIGH | `stock_quantity ≤ reorder_level` |
| `STOCKOUT_RISK` | CRITICAL | on-hand < half of sales volume |
| `EXPIRED_BEFORE_LAST_ORDER` | MEDIUM | `expiration_date < last_order_date` (data anomaly) |
| `BACKORDERED` | HIGH | `status = 'Backordered'` |
| `DISCONTINUED_WITH_STOCK` | LOW | discontinued but stock remains |
| `MISSING_CATEGORY` | LOW | category was null in source |

In [8]:
docs = etl.build_supplier_audit_docs(clean)
print(f'{len(docs)} supplier audit documents')
sample = docs[0]
print(json.dumps({**sample, 'audit_events': sample['audit_events'][:2]}, indent=2))

[TRANSFORM] Built 350 supplier audit documents (1839 events).


350 supplier audit documents
{
  "_id": "Abata",
  "supplier_name": "Abata",
  "source": "grocery_etl.py",
  "generated_at": "2026-09-21T06:38:06.750077+00:00",
  "summary": {
    "supplier_ids": [
      "17-963-6051",
      "19-809-5654",
      "40-260-8547"
    ],
    "products_supplied": 3,
    "total_stock": 215,
    "total_sales_volume": 231,
    "avg_unit_price": 6.5,
    "status_counts": {
      "Active": 1,
      "Backordered": 1,
      "Discontinued": 1
    },
    "event_count": 5,
    "severity_counts": {
      "MEDIUM": 3,
      "HIGH": 1,
      "LOW": 1
    }
  },
  "audit_events": [
    {
      "product_id": "05-498-7751",
      "product_name": "Cauliflower",
      "warehouse_location": "30408 5th Way",
      "event_type": "EXPIRED_BEFORE_LAST_ORDER",
      "severity": "MEDIUM",
      "detail": "expiration 2024-03-24 precedes last order 2024-07-10"
    },
    {
      "product_id": "43-153-8268",
      "product_name": "Vegetable Oil",
      "warehouse_location": "29815 Bonn

In [9]:
# Load into MongoDB: tries Atlas first when MONGODB_URI is set, falls back
# to mongomock so the notebook runs even without a live server.
import os
etl.load_env()

backend = 'Atlas'
total = etl.load_mongodb(docs, dry_run=False, full_refresh=True) \
    if os.environ.get('MONGODB_URI') else 0
if total == 0:  # no URI or unreachable -> in-memory fallback
    backend = 'mongomock'
    total = etl.load_mongodb(docs, dry_run=True, full_refresh=True)
print(f'supplier_audit_logs: {total} documents (backend={backend})')

[LOAD - MONGODB] Could not connect: The DNS query name does not exist: _mongodb._tcp.datawarehousinglab1.1kqx97d.mongodb.net.


[LOAD - MONGODB] Upserted 350 docs into 'supplier_audit_logs' (mongomock...); collection total = 350.


supplier_audit_logs: 350 documents (backend=mongomock)


## 6. Automated Replenishment & Stockout Prevention

The two warehouse views turn the snapshot into an actionable reorder queue — the project's expected outcome.

In [10]:
pd.read_sql('''
    SELECT replenishment_priority, COUNT(*) AS items,
           SUM(suggested_order_qty) AS units_to_order,
           ROUND(SUM(suggested_order_qty * unit_price), 2) AS est_spend
    FROM v_reorder_recommendations
    GROUP BY replenishment_priority
''', conn)

,replenishment_priority,items,units_to_order,est_spend
0,CRITICAL,115,6259,37281.15
1,HIGH,183,9606,52521.54


In [11]:
pd.read_sql('''
    SELECT product_name, category, stock_quantity, sales_volume,
           days_of_cover, stockout_risk
    FROM v_stockout_risk
    WHERE stockout_risk IN ('CRITICAL','HIGH')
    ORDER BY days_of_cover LIMIT 10
''', conn)

,product_name,category,stock_quantity,sales_volume,days_of_cover,stockout_risk
0,Egg (Chicken),Dairy,10,100,3.0,CRITICAL
1,Plum,Fruits & Vegetables,11,99,3.3,CRITICAL
2,Black Coffee,Beverages,11,89,3.7,CRITICAL
3,Long Grain Rice,Grains & Pulses,12,95,3.8,CRITICAL
4,Haddock,Seafood,11,86,3.8,CRITICAL
5,Sour Cream,Dairy,11,84,3.9,CRITICAL
6,Watermelon,Fruits & Vegetables,13,100,3.9,CRITICAL
7,Lime,Fruits & Vegetables,14,100,4.2,CRITICAL
8,Bread Flour,Grains & Pulses,11,77,4.3,CRITICAL
9,Whipped Cream,Dairy,14,90,4.7,CRITICAL


## 7. Findings

- **Data quality:** all 990 `Unit_Price` values carried `$` + trailing whitespace; `Catagory` header misspelled; 1 missing category; all dates needed normalizing to ISO. 0 rows quarantined.
- **Warehouse:** 990 facts → 124 products, 350 suppliers, 990 store locations; sales span 2024-02 → 2025-02.
- **Replenishment:** ~298 items sit at/below reorder level; ~27 items carry CRITICAL stockout risk (<7 days of cover).
- **Audit trail:** 350 supplier audit docs capture 1,800+ events (expired-before-order anomalies, backorders, reorder triggers).

The warehouse model supports automated replenishment (`v_reorder_recommendations`) and stockout prevention (`v_stockout_risk`), with supplier accountability tracked in MongoDB.